### PHASE 1 : Collecting data

In [2]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [3]:
conn = sqlite3.connect("headline_db.sqlite")
cursor = conn.cursor()

In [4]:
cursor.execute(
    "CREATE TABLE IF NOT EXISTS prices(ticker TEXT, date TEXT, close REAL)"
)

In [5]:
cursor.execute(
    "CREATE TABLE IF NOT EXISTS headlines(ticker TEXT, date TEXT, title TEXT, label TEXT, score REAL)"
)

In [6]:
conn.commit()

In [7]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
cursor.fetchall()

[('prices',), ('headlines',)]

In [8]:
# List to capture AI stocks—AI infrastructure and AI-powered companies.
stocks = ["NVDA", "META", "MSFT", "AMD", "GOOGL"]

In [9]:
# df is Dataframe—function which passes in stocks and date range for df.
df = yf.download(tickers=stocks, start="2023-03-15", end="2026-03-15")

[*********************100%***********************]  5 of 5 completed


In [10]:
df.head()

Price           Close                                                 \
Ticker            AMD       GOOGL        META        MSFT       NVDA   
Date                                                                   
2023-03-15  89.680000   95.322166  196.210236  259.299286  24.206573   
2023-03-16  96.599998   99.497643  203.334351  269.810394  25.518408   
2023-03-17  97.839996  100.787003  194.086899  272.965698  25.702244   
2023-03-20  96.809998  100.390282  196.269760  265.932251  25.877090   
2023-03-21  95.930000  104.059944  200.585922  267.446289  26.175829   

Price            High                                                 ...  \
Ticker            AMD       GOOGL        META        MSFT       NVDA  ...   
Date                                                                  ...   
2023-03-15  90.419998   96.135444  196.240001  260.315235  24.264521  ...   
2023-03-16  96.690002  100.360514  204.157890  270.162052  25.565365  ...   
2023-03-17  98.750000  101.996996  200.327916  276.775470  26.375647  ...   
2023-03-20  96.940002  100.915936  197.807694  271.060798  26.000981  ...   
2023-03-21  99.459999  104.238469  201.965099  268.638066  26.368659  ...   

Price            Open                                                 \
Ticker            AMD       GOOGL        META        MSFT       NVDA   
Date                                                                   
2023-03-15  86.769997   92.455857  191.447607  253.965606  23.739986   
2023-03-16  89.720001   95.411413  196.716287  259.074617  24.005747   
2023-03-17  96.660004   99.438150  198.998353  271.822782  25.959016   
2023-03-20  96.300003   99.299300  196.934541  270.572365  25.592343   
2023-03-21  97.000000  100.420030  201.617818  268.520847  26.156847   

Price          Volume                                           
Ticker            AMD     GOOGL      META      MSFT       NVDA  
Date                                                            
2023-03-15   86177400  50622100  42123600  46028000  524486000  
2023-03-16  115839200  65492000  50447100  54768800  583253000  
2023-03-17   94080800  61028500  50141100  69527400  848547000  
2023-03-20   92008900  32960400  25186300  43466600  432747000  
2023-03-21   85285300  42110300  31827000  34558700  547408000  

[5 rows x 25 columns]

In [11]:
df.columns

MultiIndex([( 'Close',   'AMD'),
            ( 'Close', 'GOOGL'),
            ( 'Close',  'META'),
            ( 'Close',  'MSFT'),
            ( 'Close',  'NVDA'),
            (  'High',   'AMD'),
            (  'High', 'GOOGL'),
            (  'High',  'META'),
            (  'High',  'MSFT'),
            (  'High',  'NVDA'),
            (   'Low',   'AMD'),
            (   'Low', 'GOOGL'),
            (   'Low',  'META'),
            (   'Low',  'MSFT'),
            (   'Low',  'NVDA'),
            (  'Open',   'AMD'),
            (  'Open', 'GOOGL'),
            (  'Open',  'META'),
            (  'Open',  'MSFT'),
            (  'Open',  'NVDA'),
            ('Volume',   'AMD'),
            ('Volume', 'GOOGL'),
            ('Volume',  'META'),
            ('Volume',  'MSFT'),
            ('Volume',  'NVDA')],
           names=['Price', 'Ticker'])

In [12]:
df.shape

(752, 25)

In [13]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 752 entries, 2023-03-15 to 2026-03-13
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, AMD)     752 non-null    float64
 1   (Close, GOOGL)   752 non-null    float64
 2   (Close, META)    752 non-null    float64
 3   (Close, MSFT)    752 non-null    float64
 4   (Close, NVDA)    752 non-null    float64
 5   (High, AMD)      752 non-null    float64
 6   (High, GOOGL)    752 non-null    float64
 7   (High, META)     752 non-null    float64
 8   (High, MSFT)     752 non-null    float64
 9   (High, NVDA)     752 non-null    float64
 10  (Low, AMD)       752 non-null    float64
 11  (Low, GOOGL)     752 non-null    float64
 12  (Low, META)      752 non-null    float64
 13  (Low, MSFT)      752 non-null    float64
 14  (Low, NVDA)      752 non-null    float64
 15  (Open, AMD)      752 non-null    float64
 16  (Open, GOOGL)    752 non-null    float64
 17  (Open, M

In [14]:
df.describe()

Price        Close                                                  \
Ticker         AMD       GOOGL        META        MSFT        NVDA   
count   752.000000  752.000000  752.000000  752.000000  752.000000   
mean    146.828324  178.303726  510.032443  406.722718  110.174772   
std      41.766704   59.267770  163.535076   64.594055   53.493754   
min      78.209999   95.322166  194.086899  259.299286   24.206573   
25%     112.002502  136.263771  335.286362  365.005562   48.840423   
50%     141.760002  164.499840  524.088867  409.369659  116.998009   
75%     169.445004  190.765556  645.484024  447.321579  146.969250   
max     264.329987  343.448242  788.148987  539.825195  207.017273   

Price         High                                                  ...  \
Ticker         AMD       GOOGL        META        MSFT        NVDA  ...   
count   752.000000  752.000000  752.000000  752.000000  752.000000  ...   
mean    149.719827  180.249596  516.432464  410.145542  111.981672  ...   
std      42.878968   60.022627  165.473902   64.914352   54.248785  ...   
min      85.480003   96.135444  196.240001  260.315235   24.264521  ...   
25%     114.395002  137.528325  337.253422  368.108863   49.556454  ...   
50%     144.289993  165.989432  532.311006  412.460406  119.131434  ...   
75%     172.049995  192.186498  652.566306  450.570331  148.926147  ...   
max     267.079987  348.754481  794.384398  552.242002  212.166717  ...   

Price         Open                                                  \
Ticker         AMD       GOOGL        META        MSFT        NVDA   
count   752.000000  752.000000  752.000000  752.000000  752.000000   
mean    146.924468  178.153442  510.100257  406.707281  110.223178   
std      41.941908   59.299621  164.119505   64.850846   53.622100   
min      79.220001   92.455857  191.447607  253.965606   23.739986   
25%     112.052500  136.271213  335.187120  364.869620   49.125130   
50%     142.730003  164.479676  524.796307  410.068714  117.272925   
75%     169.307503  190.554035  642.907274  446.764570  147.766464   
max     264.190002  347.095646  789.296390  552.023241  208.057150   

Price         Volume                                                          
Ticker           AMD         GOOGL          META          MSFT          NVDA  
count   7.520000e+02  7.520000e+02  7.520000e+02  7.520000e+02  7.520000e+02  
mean    5.305591e+07  3.181655e+07  1.695474e+07  2.381806e+07  3.364204e+08  
std     2.434066e+07  1.357632e+07  9.409263e+06  1.046421e+07  1.752086e+08  
min     7.956800e+06  1.009740e+07  4.726100e+06  5.855900e+06  6.552850e+07  
25%     3.606890e+07  2.321348e+07  1.131440e+07  1.726135e+07  1.955628e+08  
50%     4.822515e+07  2.849140e+07  1.495705e+07  2.121225e+07  3.052285e+08  
75%     6.357465e+07  3.573692e+07  1.942280e+07  2.670622e+07  4.300298e+08  
max     2.488596e+08  1.274901e+08  8.844010e+07  1.288553e+08  1.543911e+09  

[8 rows x 25 columns]

In [15]:
df.tail()

Price            Close                                                  \
Ticker             AMD       GOOGL        META        MSFT        NVDA   
Date                                                                     
2026-03-09  202.679993  306.359985  646.836182  409.410004  182.640106   
2026-03-10  203.229996  307.040009  653.510498  405.760010  184.760010   
2026-03-11  204.830002  308.700012  654.299805  404.880005  186.029999   
2026-03-12  197.740005  303.549988  637.634094  401.859985  183.139999   
2026-03-13  193.389999  302.279999  613.184998  395.549988  180.250000   

Price             High                                                  ...  \
Ticker             AMD       GOOGL        META        MSFT        NVDA  ...   
Date                                                                    ...   
2026-03-09  202.970001  306.799988  647.195859  410.209991  182.900102  ...   
2026-03-10  206.589996  309.510010  659.735149  410.200012  186.429918  ...   
2026-03-11  209.210007  311.420013  658.556170  409.010010  187.619995  ...   
2026-03-12  203.619995  308.940002  652.940997  406.119995  184.940002  ...   
2026-03-13  199.679993  307.690002  628.631733  404.799988  186.089996  ...   

Price             Open                                                  \
Ticker             AMD       GOOGL        META        MSFT        NVDA   
Date                                                                     
2026-03-09  189.360001  294.359985  634.236984  404.920013  176.820429   
2026-03-10  202.509995  306.170013  653.000925  410.029999  182.390128   
2026-03-11  205.110001  306.750000  654.199915  405.570007  185.910004   
2026-03-12  202.830002  306.820007  648.195060  404.630005  184.050003   
2026-03-13  198.110001  307.010010  623.356281  401.000000  184.919998   

Price         Volume                                           
Ticker           AMD     GOOGL      META      MSFT       NVDA  
Date                                                           
2026-03-09  38877200  29312100  13489700  30131900  177213600  
2026-03-10  29139200  23239700   9859300  31706400  179118500  
2026-03-11  23076400  24125700   8977200  25512100  145280400  
2026-03-12  28886900  24928300  11617500  27263900  155762700  
2026-03-13  27561900  23693100  18957600  26848000  160988400  

[5 rows x 25 columns]

In [16]:
df["Low"]

Ticker,AMD,GOOGL,META,MSFT,NVDA
Date,,,,,
2023-03-15,86.220001,91.880610,189.354036,253.213401,23.339341
2023-03-16,89.019997,94.717154,194.563185,257.189273,23.872863
2023-03-17,95.940002,99.279458,193.908292,269.927659,25.645293
2023-03-20,92.900002,99.051349,192.132231,263.607305,25.107771
2023-03-21,93.639999,100.390277,196.408696,263.284832,25.358553
...,...,...,...,...,...
2026-03-09,189.020004,294.079987,626.243828,403.500000,175.550494
2026-03-10,202.199997,305.570007,648.444828,402.929993,182.000149
2026-03-11,203.630005,305.920013,647.805365,401.589996,184.449997


In [17]:
df["Low"]["NVDA"]

Date
2023-03-15     23.339341
2023-03-16     23.872863
2023-03-17     25.645293
2023-03-20     25.107771
2023-03-21     25.358553
                 ...    
2026-03-09    175.550494
2026-03-10    182.000149
2026-03-11    184.449997
2026-03-12    181.750000
2026-03-13    179.940002
Name: NVDA, Length: 752, dtype: float64

In [18]:
(df == 0).any().sum()

np.int64(0)

In [19]:
df["Close"]
close_prices = df["Close"]

In [20]:
yf.Ticker("NVDA").news

[{'id': '16708602-ff63-496b-84b7-6d6bbc9bcf45',
  'content': {'id': '16708602-ff63-496b-84b7-6d6bbc9bcf45',
   'contentType': 'STORY',
   'title': 'Nvidia CEO Jensen Huang says company now has zero market share in China',
   'description': '',
   'summary': "Nvidia CEO Jensen Huang says the company's market share in China has fallen to zero.",
   'pubDate': '2026-05-04T15:08:05Z',
   'displayTime': '2026-05-04T15:08:05Z',
   'isHosted': True,
   'bypassModal': False,
   'previewUrl': None,
   'thumbnail': {'originalUrl': 'https://s.yimg.com/os/creatr-uploaded-images/2024-12/6f05a5f0-bc88-11ef-b2d4-0a5d1a1a4abe',
    'originalWidth': 5500,
    'originalHeight': 3667,
    'caption': '',
    'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/JWolZpsuJt_uj1rpKuG3lQ--~B/aD0zNjY3O3c9NTUwMDthcHBpZD15dGFjaHlvbg--/https://media-mbst-pub-ue1.s3.amazonaws.com/creatr-uploaded-images/2024-12/6f05a5f0-bc88-11ef-b2d4-0a5d1a1a4abe',
      'width': 5500,
      'height': 3667,
      'tag': 'orig

In [21]:
headlines = []

for stock in stocks:
    news = yf.Ticker(stock).news
    for article in news:
        headlines.append({
            "ticker": stock,
            "title": article["content"]["title"],
            "date": article["content"]["pubDate"]
        })

In [22]:
# Show all content and metadata for single headline
yf.Ticker("NVDA").news[0]

{'id': '16708602-ff63-496b-84b7-6d6bbc9bcf45',
 'content': {'id': '16708602-ff63-496b-84b7-6d6bbc9bcf45',
  'contentType': 'STORY',
  'title': 'Nvidia CEO Jensen Huang says company now has zero market share in China',
  'description': '',
  'summary': "Nvidia CEO Jensen Huang says the company's market share in China has fallen to zero.",
  'pubDate': '2026-05-04T15:08:05Z',
  'displayTime': '2026-05-04T15:08:05Z',
  'isHosted': True,
  'bypassModal': False,
  'previewUrl': None,
  'thumbnail': {'originalUrl': 'https://s.yimg.com/os/creatr-uploaded-images/2024-12/6f05a5f0-bc88-11ef-b2d4-0a5d1a1a4abe',
   'originalWidth': 5500,
   'originalHeight': 3667,
   'caption': '',
   'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/JWolZpsuJt_uj1rpKuG3lQ--~B/aD0zNjY3O3c9NTUwMDthcHBpZD15dGFjaHlvbg--/https://media-mbst-pub-ue1.s3.amazonaws.com/creatr-uploaded-images/2024-12/6f05a5f0-bc88-11ef-b2d4-0a5d1a1a4abe',
     'width': 5500,
     'height': 3667,
     'tag': 'original'},
    {'url':

In [23]:
headlines[0]

{'ticker': 'NVDA',
 'title': 'Nvidia CEO Jensen Huang says company now has zero market share in China',
 'date': '2026-05-04T15:08:05Z'}

In [24]:
len(headlines)

50

In [25]:
# Flatten data

headline_data = pd.DataFrame(headlines)

In [26]:
headline_data.head()

,ticker,title,date
0,NVDA,Nvidia CEO Jensen Huang says company now has z...,2026-05-04T15:08:05Z
1,NVDA,Big Tech pulled into growing AI capacity concerns,2026-05-04T16:58:32Z
2,NVDA,AI Chipmaker Cerebras Targets Raising Up To $4...,2026-05-04T16:50:27Z
3,NVDA,SIVR Beats SLV. Here Is Why.,2026-05-04T16:50:00Z
4,NVDA,Cerebras Systems seeks up to $26.6B valuation ...,2026-05-04T16:49:00Z


### PHASE 2 : Sentiment Scoring

In [27]:
from src.sentiment import analyze_sentiment

In [28]:
label_results = []
score_results = []

for title in headline_data["title"]:
    sentiment = analyze_sentiment(title)
    label_results.append(sentiment[0][0]["label"])
    score_results.append(sentiment[0][0]["score"])

In [29]:
headline_data["label"] = label_results
headline_data["score"] = score_results
headline_data

,ticker,title,date,label,score
0,NVDA,Nvidia CEO Jensen Huang says company now has z...,2026-05-04T15:08:05Z,neutral,0.755732
1,NVDA,Big Tech pulled into growing AI capacity concerns,2026-05-04T16:58:32Z,negative,0.987875
2,NVDA,AI Chipmaker Cerebras Targets Raising Up To $4...,2026-05-04T16:50:27Z,positive,0.999280
3,NVDA,SIVR Beats SLV. Here Is Why.,2026-05-04T16:50:00Z,neutral,0.999886
4,NVDA,Cerebras Systems seeks up to $26.6B valuation ...,2026-05-04T16:49:00Z,positive,0.998417
5,NVDA,Did Anthropic Suck the Air From Palantir’s Com...,2026-05-04T16:45:21Z,negative,0.546641
6,NVDA,AMD Sinks 6% Despite a Holding Pattern in Inte...,2026-05-04T16:27:42Z,negative,0.997712
7,NVDA,Wall Street Sees 21% Upside for Rivian Despite...,2026-05-04T16:23:07Z,positive,0.998614
8,NVDA,Jim Cramer Just Won’t Give Up on NVIDIA (NVDA),2026-05-04T16:20:58Z,neutral,0.996618
9,NVDA,Intel's Stock Hits a New All-Time High: Is It ...,2026-05-04T16:20:00Z,positive,0.999339


In [30]:
df.head()

Price           Close                                                 \
Ticker            AMD       GOOGL        META        MSFT       NVDA   
Date                                                                   
2023-03-15  89.680000   95.322166  196.210236  259.299286  24.206573   
2023-03-16  96.599998   99.497643  203.334351  269.810394  25.518408   
2023-03-17  97.839996  100.787003  194.086899  272.965698  25.702244   
2023-03-20  96.809998  100.390282  196.269760  265.932251  25.877090   
2023-03-21  95.930000  104.059944  200.585922  267.446289  26.175829   

Price            High                                                 ...  \
Ticker            AMD       GOOGL        META        MSFT       NVDA  ...   
Date                                                                  ...   
2023-03-15  90.419998   96.135444  196.240001  260.315235  24.264521  ...   
2023-03-16  96.690002  100.360514  204.157890  270.162052  25.565365  ...   
2023-03-17  98.750000  101.996996  200.327916  276.775470  26.375647  ...   
2023-03-20  96.940002  100.915936  197.807694  271.060798  26.000981  ...   
2023-03-21  99.459999  104.238469  201.965099  268.638066  26.368659  ...   

Price            Open                                                 \
Ticker            AMD       GOOGL        META        MSFT       NVDA   
Date                                                                   
2023-03-15  86.769997   92.455857  191.447607  253.965606  23.739986   
2023-03-16  89.720001   95.411413  196.716287  259.074617  24.005747   
2023-03-17  96.660004   99.438150  198.998353  271.822782  25.959016   
2023-03-20  96.300003   99.299300  196.934541  270.572365  25.592343   
2023-03-21  97.000000  100.420030  201.617818  268.520847  26.156847   

Price          Volume                                           
Ticker            AMD     GOOGL      META      MSFT       NVDA  
Date                                                            
2023-03-15   86177400  50622100  42123600  46028000  524486000  
2023-03-16  115839200  65492000  50447100  54768800  583253000  
2023-03-17   94080800  61028500  50141100  69527400  848547000  
2023-03-20   92008900  32960400  25186300  43466600  432747000  
2023-03-21   85285300  42110300  31827000  34558700  547408000  

[5 rows x 25 columns]

### Phase 3 : Analyze data

In [31]:
# Reshape data

reshaped_data = df["Close"].stack().reset_index().rename(columns={0: "close", "Ticker": "ticker", "Date": "date"})
reshaped_data.head()

,date,ticker,close
0,2023-03-15,AMD,89.680000
1,2023-03-15,GOOGL,95.322166
2,2023-03-15,META,196.210236
3,2023-03-15,MSFT,259.299286
4,2023-03-15,NVDA,24.206573


In [32]:
reshaped_data = reshaped_data.set_index("date")

In [33]:
reshaped_data.to_sql("prices", conn, if_exists="replace", index=True)

3760

In [65]:
headline_data.to_sql("headlines", conn, if_exists="replace", index=True)

50

In [66]:
cursor.execute("SELECT COUNT(*) FROM prices")
print(cursor.fetchone())

cursor.execute("SELECT COUNT(*) FROM headlines")
print(cursor.fetchone())

(3760,)
(50,)


In [34]:
# Group data by months

# reshaped_data.dtypes

reshaped_data.groupby(["ticker", pd.Grouper(freq="ME")])["close"].mean()

ticker  date      
AMD     2023-03-31     96.601537
        2023-04-30     90.810001
        2023-05-31    102.217273
        2023-06-30    117.791905
        2023-07-31    113.690000
                         ...    
NVDA    2025-11-30    187.962520
        2025-12-31    182.324673
        2026-01-31    186.711894
        2026-02-28    185.701528
        2026-03-31    182.350104
Name: close, Length: 185, dtype: float64

In [35]:
# Explore monthly trends

monthly_close = reshaped_data.groupby(["ticker", pd.Grouper(freq="ME")])["close"].mean()
monthly_close.head()

ticker  date      
AMD     2023-03-31     96.601537
        2023-04-30     90.810001
        2023-05-31    102.217273
        2023-06-30    117.791905
        2023-07-31    113.690000
Name: close, dtype: float64

In [36]:
reshaped_data.groupby(["ticker", pd.Grouper(freq="QE")])["close"].mean()

ticker  date      
AMD     2023-03-31     96.601537
        2023-06-30    103.996775
        2023-09-30    108.554127
        2023-12-31    117.857460
        2024-03-31    174.810654
                         ...    
NVDA    2025-03-31    126.703980
        2025-06-30    125.817829
        2025-09-30    174.281514
        2025-12-31    186.121490
        2026-03-31    185.429958
Name: close, Length: 65, dtype: float64

In [37]:
# Explore quarterly trends

quarterly_close = reshaped_data.groupby(["ticker", pd.Grouper(freq="QE")])["close"].mean()
quarterly_close.head()

ticker  date      
AMD     2023-03-31     96.601537
        2023-06-30    103.996775
        2023-09-30    108.554127
        2023-12-31    117.857460
        2024-03-31    174.810654
Name: close, dtype: float64

In [38]:
headline_data.groupby(["ticker", "label"])["label"].count()

ticker  label   
AMD     negative    4
        neutral     1
        positive    5
GOOGL   negative    2
        neutral     5
        positive    3
META    neutral     8
        positive    2
MSFT    negative    2
        neutral     3
        positive    5
NVDA    negative    3
        neutral     3
        positive    4
Name: label, dtype: int64

In [39]:
sentiment_count = headline_data.groupby(["ticker", "label"])["label"].count()

In [40]:
headline_data.groupby(["ticker"])["score"].mean()

ticker
AMD      0.993738
GOOGL    0.995332
META     0.995715
MSFT     0.956625
NVDA     0.928011
Name: score, dtype: float64

In [41]:
score_avg = headline_data.groupby(["ticker"])["score"].mean()

In [42]:
first_close = quarterly_close.groupby("ticker").first()
last_close = quarterly_close.groupby("ticker").last()

In [43]:
((last_close - first_close) / first_close) * 100

ticker
AMD      123.770470
GOOGL    214.705770
META     224.733361
MSFT      57.974528
NVDA     603.180094
Name: close, dtype: float64

In [44]:
stock_roc = ((last_close - first_close) / first_close) * 100

In [45]:
# All my analysis variables

# sentiment_count — label counts per ticker
# score_avg — average confidence score per ticker
# quarterly_close — price trend over 12 quarterly_close
# stock_roc — overall % price change over 3 years

In [46]:
headline_data[headline_data["score"] < 0.80]

,ticker,title,date,label,score
0,NVDA,Nvidia CEO Jensen Huang says company now has z...,2026-05-04T15:08:05Z,neutral,0.755732
5,NVDA,Did Anthropic Suck the Air From Palantir’s Com...,2026-05-04T16:45:21Z,negative,0.546641
25,MSFT,Microsoft Pentagon AI Deal Deepens Cloud Ties ...,2026-05-04T16:13:43Z,negative,0.614907


In [47]:
low_confidence = headline_data[headline_data["score"] < 0.80]

### Phase 4 : Visualize data

In [48]:
quarterly_df = quarterly_close.reset_index()

In [49]:
px.line(
    data_frame=quarterly_df, 
    x="date", 
    y="close", 
    color="ticker", 
    title="Q1 '23—Q2 '26 Quarterly Stock Prices"
    )

In [50]:
sentiment_df = sentiment_count.reset_index(name="count")

In [51]:
# Stacked bar chart
# px.bar(data_frame=sentiment_df, x="ticker", y="count", color="label", title="Recent News Headlines Sentiment")

In [52]:
px.bar(
    data_frame=sentiment_df,
    x="ticker", 
    y="count", 
    color="label", 
    barmode="group",
    title="Recent News Headlines Sentiment", 
    color_discrete_map = {
        "positive": "#48a860", 
        "neutral": "#bebdb8", 
        "negative": "#cd5c5c"
        }
    )

In [53]:
price_chart = px.line(
    data_frame=quarterly_df, 
    x="date", 
    y="close", 
    color="ticker", 
    title="Q1 '23—Q2 '26 Quarterly Stock Prices"
    )

sentiment_chart = px.bar(
    data_frame=sentiment_df,
    x="ticker", 
    y="count", 
    color="label", 
    barmode="group",
    title="Recent News Headlines Sentiment", 
    color_discrete_map = {
        "positive": "#48a860", 
        "neutral": "#bebdb8", 
        "negative": "#cd5c5c"
        }
    )

In [54]:
price_chart.write_html("outputs/price_chart.html")
sentiment_chart.write_html("outputs/sentiment_chart.html")

In [55]:
# Creates empty subplot container

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Quarterly Close Price (2023-2026)", "Current News Sentiment by Ticker")
    )

In [56]:
for trace in price_chart.data:
    fig.add_trace(trace, row=1, col=1)

for trace in sentiment_chart.data:
    fig.add_trace(trace, row=2, col=1)

In [57]:
fig.update_layout(
    title_text="AI Stock Performance & Current Sentiment Analysis",
    height=800
)

In [58]:
fig.write_html("outputs/stock_dashboard.html")

In [59]:
quarterly_close.index.get_level_values("date").max()

Timestamp('2026-03-31 00:00:00')

In [60]:
last_close

ticker
AMD      216.165714
GOOGL    318.872785
META     653.582659
MSFT     427.593246
NVDA     185.429958
Name: close, dtype: float64

In [61]:
stock_roc

ticker
AMD      123.770470
GOOGL    214.705770
META     224.733361
MSFT      57.974528
NVDA     603.180094
Name: close, dtype: float64

In [62]:
for ticker in last_close.index:
    fig.add_annotation(
        x = "2026-03-31",
        y = last_close[ticker],
        text = f"+{stock_roc[ticker]:.1f}%",
        font = dict(size=10)
    )

In [63]:
fig

In [64]:
fig.write_html("outputs/stock_dashboard.html")